In [ ]:
# 본인의 이름, 학번을 작성하여 실행하세요.
print("김희래")
print("20211873")

In [10]:
class Foo:
    def __getitem__(self, pos):
        return range(0, 30, 10)[pos]

In [15]:
f[1]

10

In [12]:
f = Foo()

In [14]:
for i in f: print(i)

0
10
20


In [16]:
20 in f

True

In [17]:
15 in f

False

In [39]:
%%writefile frenchdeck.py
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]
    def __len__(self):
        return len(self._cards)
    def __getitem__(self, position):
        return self._cards[position]

Overwriting frenchdeck.py


In [40]:
from random import shuffle
from frenchdeck import FrenchDeck
deck = FrenchDeck()
shuffle(deck)

In [41]:
def set_card(deck, position, card):
    deck._cards[position] = card

FrenchDeck.__setitem__ = set_card
shuffle(deck)
deck[:5]

[Card(rank='6', suit='diamonds'),
 Card(rank='7', suit='diamonds'),
 Card(rank='10', suit='spades'),
 Card(rank='J', suit='spades'),
 Card(rank='Q', suit='clubs')]

In [47]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDecK2(collections.abc.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]
    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, position, value):
        self._cards[position] = value

    def __delattr__(self, position):
        del self.card[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

In [43]:
%%writefile tombola.py
import abc

class Tombola(abc.ABC):

    @abc.abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abc.abstractmethod
    def pick(self):
        """Remove item at random, returning it.

        This method should raise `LookupError` when the instance is empty.
        """

    def loaded(self):
        """Return `True` if there's at least 1 item, `False` otherwise."""
        return bool(self.inspect())

    def inspect(self):
        """Return a sorted tuple with the items currently inside."""
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))

Overwriting tombola.py


In [48]:
import random

from tombola import Tombola

class BingoCage(Tombola):

    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        self.pick()


In [46]:
from random import randrange

from tombola import Tombola

@Tombola.register
class LotteryBlower(Tombola):

    def pick(self):
        if self._balls:
            position = randrange(len(self._balls))
            return self._balls.pop(position)
        else:
            raise LookupError('pick from empty BingoCage')

    load = list.extend

    def loaded(self):
        return bool(self._balls)

    def inspect(self):
        return tuple(sorted(self))